# CSI 1000 Recent-Market RL Research

FTShare-only, Backward-adjusted daily data. No performance is claimed until every training and evaluation cell succeeds.


In [ ]:
!git clone -q https://github.com/ThisizHTZ/ftshare_mo.git /content/ftshare_mo 2>/dev/null || git -C /content/ftshare_mo pull -q
%cd /content/ftshare_mo/rl_research
!pip -q install -r requirements.txt
!pip -q install -e .


## Optional persistent cache
Set USE_DRIVE to True to preserve FTShare downloads and training artifacts across Colab sessions.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, pathlib
    target = pathlib.Path('/content/drive/MyDrive/csi1000_rl_artifacts')
    target.mkdir(parents=True, exist_ok=True)
    local = pathlib.Path('artifacts')
    if local.exists() and not local.is_symlink(): os.system('rm -rf artifacts')
    if not local.exists(): local.symlink_to(target, target_is_directory=True)


## Lightweight validation
These tests use synthetic data only and do not claim market performance.


In [ ]:
!pytest -q


## Download and quality gate
Only FTShare 000852.XSHG daily bars with adjust=Backward are requested in yearly chunks.


In [ ]:
!python -m csi1000_rl.pipeline download
import json
quality = json.load(open('artifacts/data/processed/data_quality.json'))
assert quality['is_valid'], quality
quality


## Train, validate, and test
Validation selects seeds. The final six months remain isolated for out-of-sample evaluation.


In [ ]:
!python -m csi1000_rl.pipeline train


## Inspect and package real outputs
The completed report is created only after successful real training and includes data/config hashes.


In [ ]:
from IPython.display import display, HTML
display(HTML(filename='artifacts/results/report_zh.html'))
!zip -qr csi1000_rl_results.zip artifacts/results artifacts/data/processed/data_quality.json config/default.yaml
from google.colab import files
files.download('csi1000_rl_results.zip')
